<a href="https://colab.research.google.com/github/KeertanPatel-CS/movie_recommender-svd/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System
## Milestone 4: Hidden Factors and Personal Recommendations

**Team members:** Kirtan Patel and Rudra Patel  
**Course:** MATH 344L

We explore the patterns learned by Model 2 and use personal movie
ratings to generate recommendations.

We compare the personalized predictions with the baseline and examine
which movies receive the largest positive and negative adjustments.

In [ ]:
import os
import sys
import urllib.request
import zipfile
import numpy as np
import pandas as pd

# Download the project's reusable code into this Colab session.
REPO = "movie_recommender-svd"

if not os.path.exists(REPO):
    !git clone -q https://github.com/KeertanPatel-CS/movie_recommender-svd.git

# Import the functions saved in our repository.
sys.path.insert(0, f"{REPO}/src")

from evaluate import to_matrix
from models import baseline_matrix, rank_k

print("Project functions imported successfully.")

Project functions imported successfully.


This cell imports the Python libraries and our reusable project functions.
We use the same baseline and low-rank reconstruction methods as before.

In [ ]:
# Download MovieLens if the ratings file is not already available.
if not os.path.exists("ml-100k/u.data"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip"
    )

    with zipfile.ZipFile("ml-100k.zip", "r") as zip_ref:
        zip_ref.extractall()

# Load the user ratings.
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"]
)

# Load movie titles in movie-ID order.
movie_info = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item", "title"]
).sort_values("item")

titles = movie_info["title"].to_numpy()

print("Number of ratings:", len(ratings))
print("Number of movies:", len(titles))
print("Movie 50:", titles[49])

Number of ratings: 100000
Number of movies: 1682
Movie 50: Star Wars (1977)


The ratings table contains one row per observed rating.
The titles array connects movie positions with readable names.

Movie IDs start at 1, but array positions start at 0.
For example, movie 50 is stored at titles[49].

In [ ]:
# Fit Model 2 on all available ratings for interpretation.
R = to_matrix(ratings)
known = ~np.isnan(R)

B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)

# Keep these factors separate from later personalized-model factors.
U_all, s_all, Vt_all = np.linalg.svd(
    X, full_matrices=False
)

# Examine movies with at least 50 ratings.
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50

# Use the rank chosen through validation in Milestone 3.
chosen_k = 15

print("Matrix shape:", R.shape)
print("Movies with at least 50 ratings:", popular.sum())
print("Selected rank:", chosen_k)

Matrix shape: (943, 1682)
Movies with at least 50 ratings: 603
Selected rank: 15


In Milestone 3, we held out ratings to evaluate prediction accuracy.
Here, we fit on all available ratings to interpret the model and
generate recommendations. We do not report test accuracy from this fit.

We subtract the baseline before computing SVD. This lets the factors
describe patterns beyond general user and movie effects.

We inspect movies with at least 50 ratings so that our interpretation
is less influenced by movies with very few observations.

In [ ]:
def show_factor(f, n=6):
    """Display popular movies at both ends of a hidden factor."""

    # Factor numbering starts at 1; array indexing starts at 0.
    values = Vt_all[f - 1]

    # Sort eligible movies by their score on this factor.
    movie_positions = np.where(popular)[0]
    order = movie_positions[np.argsort(values[movie_positions])]

    print(f"\nFactor {f}: lowest scores")
    for movie in order[:n]:
        print(f"  {values[movie]:+.3f}  {titles[movie]}")

    print(f"Factor {f}: highest scores")
    for movie in order[::-1][:n]:
        print(f"  {values[movie]:+.3f}  {titles[movie]}")


# Inspect all three factors required by Exercise 1.
show_factor(1)
show_factor(2)
show_factor(3)


Factor 1: lowest scores
  -0.087  Clockwork Orange, A (1971)
  -0.079  Trainspotting (1996)
  -0.072  Leaving Las Vegas (1995)
  -0.072  Pulp Fiction (1994)
  -0.069  Brazil (1985)
  -0.067  Fargo (1996)
Factor 1: highest scores
  +0.211  Independence Day (ID4) (1996)
  +0.175  Return of the Jedi (1983)
  +0.149  Raiders of the Lost Ark (1981)
  +0.149  Back to the Future (1985)
  +0.136  Braveheart (1995)
  +0.134  Forrest Gump (1994)

Factor 2: lowest scores
  -0.099  Independence Day (ID4) (1996)
  -0.092  Twister (1996)
  -0.077  Eraser (1996)
  -0.074  Air Force One (1997)
  -0.067  Saint, The (1997)
  -0.066  Phenomenon (1996)
Factor 2: highest scores
  +0.203  Star Wars (1977)
  +0.199  Pulp Fiction (1994)
  +0.173  Godfather, The (1972)
  +0.168  Empire Strikes Back, The (1980)
  +0.151  Fargo (1996)
  +0.146  Return of the Jedi (1983)

Factor 3: lowest scores
  -0.137  Independence Day (ID4) (1996)
  -0.131  Star Trek: First Contact (1996)
  -0.130  Rock, The (1996)
  -0.129 

Each row of Vt_all assigns movies a score on one hidden factor.
We inspect the movies with the lowest and highest scores to look
for a possible interpretation.

A positive score does not mean a movie is better. The two ends
represent contrasting patterns in ratings.

SVD factor signs can flip without changing the reconstructed matrix.
Therefore, the groups of movies matter more than which group has
positive scores. Any names we give the factors are interpretations,
not labels supplied to the model.